# 05 · Features (Nico)

**Starting point:** `features_ludovico`. Of his 8 feature groups only the **month of purchase** (`calendar`) helped, and every group was built from **its own row**.

**Question of this notebook:** does the customer's **purchase history**, read from the earlier rows of the same customer in train and test, add something the row alone does not have? And does a better ROC AUC reach the Kaggle metric (F1), or is it lost in the choice of the threshold?

**Method:** the same as `features_ludovico`, so the numbers are comparable: same 8 half-year folds (`prep_folds`), same two measuring instruments (logistic regression with the linear recipe, gradient boosting with the tree recipe), paired differences against the reference, SEM and folds won. The **reference is Ludovico's result**: every column + the month.

**Leakage rule:** a feature of a purchase may only use purchases made **before** it. Nothing is learned from the target. A purchase's *later* rows are never read: they would reveal the target itself (99% of train rows have target 1 exactly when the same customer has another row within 90 days).

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. The missing period between train and test</font>](#missing)
* [<font color='#E8800A'>2. The new feature groups</font>](#groups)
* [<font color='#E8800A'>3. Each group added to the reference</font>](#alone)
* [<font color='#E8800A'>4. Effect on the official validation</font>](#official)
* [<font color='#E8800A'>5. From ROC AUC to F1: the threshold</font>](#threshold)
* [<font color='#E8800A'>6. Summary</font>](#summary)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `preprocessing_ludovico.ipynb` (and, through it, validation and cleaning): clean data, `prep_folds`, `official_folds`, the two recipes, `paired_table`, `evaluate`, `best_threshold`. About 2 minutes. The pieces of `features_ludovico` needed here (`FeatureBuilder`, the `calendar` group, `compare`) are restated below exactly as they are defined there, as `submission_v03_v04_ludovico` does.

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_validate

X_full = train.drop(columns=[TARGET])
X_test_full = test.copy()
ALWAYS_DROP = ["customer_id", "purchase_date", "random_noise"]
SHORT = 90  # the target's horizon in days

In [3]:
# --- restated from features_ludovico §1 ---
FEATURE_GROUPS = {}


def feature_group(name):
    """Register a function df -> df that adds columns (no learning from the target)."""
    def register(func):
        FEATURE_GROUPS[name] = func
        return func
    return register


class FeatureBuilder(BaseEstimator, TransformerMixin):
    """Pipeline step: add the chosen feature groups, then drop identifiers and the listed columns."""

    def __init__(self, groups=(), drop=()):
        self.groups = groups
        self.drop = drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        for name in self.groups:
            X = FEATURE_GROUPS[name](X)
        return X.drop(columns=[c for c in [*ALWAYS_DROP, *self.drop] if c in X.columns])


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


RECIPES = {
    "logistic regression": lambda: linear_recipe(LogisticRegression(max_iter=2000)).set_params(
        prep__num__transform=FunctionTransformer(signed_log1p, feature_names_out="one-to-one")),
    "gradient boosting": lambda: tree_recipe(HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
}


def feature_pipeline(config, model_name):
    """FeatureBuilder(config) followed by the preprocessing recipe and the model of `model_name`."""
    return Pipeline([("features", FeatureBuilder(**config)), *RECIPES[model_name]().steps])


def compare(configs, model_name, reference, folds=None):
    """Score every configuration on the same folds (ROC AUC) and return the paired table."""
    folds = prep_folds if folds is None else folds
    scores = {name: cross_validate(feature_pipeline(config, model_name), X_full, y, cv=folds,
                                   scoring="roc_auc", n_jobs=-1)["test_score"]
              for name, config in configs.items()}
    return paired_table(pd.DataFrame(scores, index=pd.RangeIndex(1, len(folds) + 1, name="fold")), reference)


@feature_group("calendar")
def calendar(df):
    return df.assign(purchase_month=df["purchase_date"].dt.month.astype(str))  # categorical: one-hot

# <font color='#E8800A'>1. The missing period between train and test</font> <a class="anchor" id="missing"></a>
[Back to TOC](#toc)

**The idea:** put train and test together, sort each customer's purchases by date, and look at the **pauses** between one purchase and the next (in days). A customer whose past pauses were almost all ≤ 90 days probably buys again within 90 days.

**The catch:** the purchases between the end of train (2025-02-15) and the start of test (2025-05-24) are **not in any file**. Each row also carries `days_since_previous_purchase`, which the organisers computed on the complete history. So for each pause we can check whether the purchase before it is really in our data: the pause we measure must match that column.

In [4]:
rows = (pd.concat([X_full.assign(file="train"), X_test_full.assign(file="test")])
        [["customer_id", "purchase_date", "days_since_previous_purchase", "file"]]
        .sort_values(["customer_id", "purchase_date"], kind="stable"))
rows["pause"] = rows.groupby("customer_id")["purchase_date"].diff().dt.days
# The pause we measure is real when it matches the organisers' column (one day of tolerance)
rows["real_pause"] = (rows["pause"] - rows["days_since_previous_purchase"]).abs() <= 1

first_test = rows[rows["file"] == "test"].groupby("customer_id").head(1)
later = rows[rows["pause"].notna() & rows["days_since_previous_purchase"].notna()]
pd.Series({
    "train rows: measured pause = official one": later.loc[later["file"] == "train", "real_pause"].mean(),
    "later test rows: measured pause = official one": later.loc[later["file"] == "test"].drop(first_test.index, errors="ignore")["real_pause"].mean(),
    "FIRST test purchase of each customer: measured = official": first_test.dropna(subset=["pause"])["real_pause"].mean(),
}, name="share").to_frame().style.format("{:.1%}")

,share
train rows: measured pause = official one,99.1%
later test rows: measured pause = official one,100.0%
FIRST test purchase of each customer: measured = official,47.3%


In [5]:
# One customer whose first test purchase follows purchases that are missing from the files
example = first_test[~first_test["real_pause"] & first_test["pause"].notna()
                     & (first_test["days_since_previous_purchase"] < 60)].iloc[0]["customer_id"]
example_rows = rows[rows["customer_id"] == example]
position = example_rows.index.get_loc(first_test.index[first_test["customer_id"] == example][0])
example_rows[["file", "purchase_date", "pause", "days_since_previous_purchase"]].iloc[position - 2:position + 2]

,file,purchase_date,pause,days_since_previous_purchase
ID,,,,
10004615,train,2025-01-11,29.0,29.0
10006088,train,2025-02-08,28.0,28.0
10001522,test,2025-06-21,133.0,49.0
10000156,test,2025-07-18,27.0,27.0


**Reading the result**
- Inside train, and inside test, the pause we measure matches the official column almost always. The few exceptions are the 65 empty train rows dropped in cleaning (purchases whose date is lost).
- For the **first test purchase of each customer** it matches in only about 4 cases out of 10. In the example the files say the previous purchase was in February (133 days), the official column says 49 days: the customer also bought around early May, a purchase that is not in the files.
- **Rule:** a measured pause enters the history **only if it matches** `days_since_previous_purchase`. A pause that jumps over missing purchases is left out. The rule is the same for train and test and needs no hand-written date.

# <font color='#E8800A'>2. The new feature groups</font> <a class="anchor" id="groups"></a>
[Back to TOC](#toc)

| Group | New columns | Hypothesis |
|---|---|---|
| `history` | number of real past pauses; share of them ≤ 90 days; their median; the mean of the last 3 | the existing `prior_mean_gap_days` is a mean, skewed by one long break. The **share of pauses ≤ 90 days** asks directly *how often did this customer come back within 90 days*, which is the question of the target |
| `next_peak` | days from the purchase to the start of the next peak month (March, July, November) | Ludovico's 4-month cycle as **one number**: if a peak begins within 90 days the customer is likely to order again in it. 12 one-hot months become a single column, which suits a linear model |
| `seasonal` | purchases of the customer in the 90 days after **the same date 1 year earlier**; share of the last 3 years with at least one purchase in that window; mean count over those windows | the month is a cycle **common to every customer**. Each customer also has **their own calendar** (some reorder every summer, some never in winter). The window one year earlier asks the target's own question at a time when the answer is already known |
| `weekday` | day of the week of the purchase (categorical) | the anonymisation keeps intervals, so the weekday pattern survives (up to a relabelling of the days). Regular orders may be placed on fixed days |

**History of a purchase = the pauses up to and including the one that ends at this purchase** (that one is `days_since_previous_purchase` itself, already known). The pause that *starts* at this purchase is the target, so it is never used. The `seasonal` windows end at least 275 days before the purchase.

In [6]:
# Real pauses only (rule of section 1); the others become NaN and are ignored by the statistics
rows["kept_pause"] = rows["pause"].where(rows["real_pause"])
rows["short"] = (rows["kept_pause"] <= SHORT).astype(float).where(rows["kept_pause"].notna())

by_customer = rows.groupby("customer_id")
HISTORY = pd.DataFrame({
    "hist_pauses": by_customer["kept_pause"].transform(lambda s: s.expanding().count()),
    "hist_share_short": by_customer["short"].transform(lambda s: s.expanding().mean()),
    "hist_median_pause": by_customer["kept_pause"].transform(lambda s: s.expanding().median()),
    "hist_recent_pause": by_customer["kept_pause"].transform(
        lambda s: s.dropna().rolling(3, min_periods=1).mean().reindex(s.index).ffill()),
}, index=rows.index)

# Check: each value depends only on the same customer's rows up to this one (dates <= this purchase)
assert HISTORY.index.is_unique and set(HISTORY.index) == set(X_full.index) | set(X_test_full.index)
HISTORY.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
hist_pauses,7455.0,25.20,39.43,0.0,4.00,12.00,29.00,274.0
hist_share_short,6818.0,0.61,0.32,0.0,0.36,0.67,0.89,1.0
hist_median_pause,6818.0,88.60,130.36,1.0,29.00,57.00,111.50,3199.0
hist_recent_pause,6818.0,102.11,141.51,1.0,35.00,69.67,121.33,3199.0


**The `seasonal` windows.** For a purchase on date *d* and *k* = 1, 2, 3 years: count the customer's purchases in (*d* − *k*·365, *d* − *k*·365 + 90]. A window is **unknown (NaN)**, not 0, when
- it starts before the customer's first purchase in the data (the customer did not exist yet), or
- it overlaps the missing period between train and test (section 1): there the files show no purchase, but the customer may have bought.

In [7]:
MISSING = (X_full["purchase_date"].max(), X_test_full["purchase_date"].min())  # no row in between
dates_of = rows.groupby("customer_id")["purchase_date"].apply(lambda s: np.sort(s.to_numpy()))
first_of = rows.groupby("customer_id")["purchase_date"].min()


def same_window(years):
    """Purchases of the customer in the 90 days after the same date `years` years earlier (NaN if unknown)."""
    starts = rows["purchase_date"] - pd.Timedelta(days=365 * years)
    counts = []
    for customer, start in zip(rows["customer_id"], starts):
        end = start + pd.Timedelta(days=SHORT)
        if start < first_of[customer] or (start < MISSING[1] and end > MISSING[0]):
            counts.append(np.nan)
            continue
        dates = dates_of[customer]
        counts.append(np.searchsorted(dates, np.datetime64(end), "right")
                      - np.searchsorted(dates, np.datetime64(start), "right"))
    return pd.Series(counts, index=rows.index, dtype=float)


windows = pd.concat({k: same_window(k) for k in [1, 2, 3]}, axis=1)
SEASONAL = pd.DataFrame({
    "season_last_year": windows[1],
    "season_share_years": (windows > 0).astype(float).where(windows.notna()).mean(axis=1),
    "season_mean_count": windows.mean(axis=1),
})
SEASONAL.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
season_last_year,5327.0,1.92,3.01,0.0,0.00,1.00,2.0,22.0
season_share_years,5594.0,0.67,0.38,0.0,0.33,0.67,1.0,1.0
season_mean_count,5594.0,1.89,2.84,0.0,0.50,1.00,2.0,19.0


In [8]:
PEAK_MONTHS = [3, 7, 11]


@feature_group("history")
def history(df):
    return df.join(HISTORY)


def next_peak_start(date):
    """First day of the first peak month after the month of `date`."""
    month = next((m for m in PEAK_MONTHS if m > date.month), PEAK_MONTHS[0])
    year = date.year + (month <= date.month)
    return pd.Timestamp(year=year, month=month, day=1)


@feature_group("next_peak")
def next_peak(df):
    return df.assign(days_to_next_peak=(df["purchase_date"].map(next_peak_start) - df["purchase_date"]).dt.days)


@feature_group("seasonal")
def seasonal(df):
    return df.join(SEASONAL)


@feature_group("weekday")
def weekday(df):
    return df.assign(weekday=df["purchase_date"].dt.dayofweek.astype(str))  # categorical: one-hot


for name in ["history", "next_peak", "seasonal", "weekday"]:
    added = FEATURE_GROUPS[name](X_full.head(50)).columns.difference(X_full.columns)
    FEATURE_GROUPS[name](X_test_full.head(50))
    print(f"{name:10s} -> {', '.join(added)}")

history    -> hist_median_pause, hist_pauses, hist_recent_pause, hist_share_short
next_peak  -> days_to_next_peak
seasonal   -> season_last_year, season_mean_count, season_share_years
weekday    -> weekday


**Do the new columns separate the target?** Repeat rate by bins of each new column (train only, for description; the models are compared in section 3).

In [9]:
described = X_full.copy()
for name in ["history", "next_peak", "seasonal", "weekday"]:
    described = FEATURE_GROUPS[name](described)
described["target"] = y
bins = {
    "hist_share_short": pd.cut(described["hist_share_short"], [-0.01, 0.25, 0.5, 0.75, 0.9, 1.0]),
    "days_to_next_peak": pd.cut(described["days_to_next_peak"], [0, 30, 60, 90, 125]),
    "season_share_years": pd.cut(described["season_share_years"], [-0.01, 0.0, 0.34, 0.67, 1.0]),
    "weekday": described["weekday"],
}
for name, groups in bins.items():
    display(described.groupby(groups, observed=True)["target"].agg(["mean", "size"]).T.round(2))
print("share of NaN:", described[["hist_share_short", "season_last_year", "season_share_years"]].isna().mean().round(3).to_dict())

hist_share_short,"(-0.01, 0.25]","(0.25, 0.5]","(0.5, 0.75]","(0.75, 0.9]","(0.9, 1.0]"
mean,0.28,0.4,0.59,0.77,0.83
size,1059.00,1175.0,1181.00,1022.00,1420.00


days_to_next_peak,"(0, 30]","(30, 60]","(60, 90]","(90, 125]"
mean,0.63,0.7,0.58,0.46
size,1267.00,1043.0,1425.00,2734.00


season_share_years,"(-0.01, 0.0]","(0.0, 0.34]","(0.34, 0.67]","(0.67, 1.0]"
mean,0.32,0.38,0.57,0.76
size,883.00,507.00,1023.00,2271.00


weekday,0,1,2,3,4,5,6
mean,0.67,0.63,0.55,0.62,0.64,0.5,0.56
size,630.00,683.00,84.00,24.00,811.00,3215.0,1022.00


share of NaN: {'hist_share_short': 0.095, 'season_last_year': 0.276, 'season_share_years': 0.276}


# <font color='#E8800A'>3. Each group added to the reference</font> <a class="anchor" id="alone"></a>
[Back to TOC](#toc)

**Reference:** `calendar`, the feature set chosen in `features_ludovico`. Each new group is added to the month on its own, then `seasonal` and `weekday` together. `next_peak instead of month` checks whether the single cycle column can replace the 12 month levels.

In [10]:
REFERENCE = "month (Ludovico)"
configs = {
    REFERENCE: {"groups": ["calendar"]},
    "month + history": {"groups": ["calendar", "history"]},
    "month + next_peak": {"groups": ["calendar", "next_peak"]},
    "month + seasonal": {"groups": ["calendar", "seasonal"]},
    "month + weekday": {"groups": ["calendar", "weekday"]},
    "month + seasonal + weekday": {"groups": ["calendar", "seasonal", "weekday"]},
    "next_peak instead of month": {"groups": ["next_peak"]},
}
tables = {model_name: compare(configs, model_name, REFERENCE) for model_name in RECIPES}
show(tables["logistic regression"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
month + seasonal + weekday,0.8022,+0.0052,0.0026,+2.0,6/8
month + seasonal,0.8006,+0.0037,0.0021,+1.7,7/8
month + weekday,0.7986,+0.0016,0.0017,+0.9,5/8
month + history,0.7981,+0.0011,0.0014,+0.8,5/8
month (Ludovico),0.7970,+0.0000,-,-,ref
month + next_peak,0.7964,-0.0006,0.0005,-1.2,4/8
next_peak instead of month,0.7930,-0.0040,0.0032,-1.2,2/8


In [11]:
show(tables["gradient boosting"])

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
month + seasonal + weekday,0.7930,+0.0074,0.0059,+1.3,6/8
month + seasonal,0.7882,+0.0026,0.0050,+0.5,6/8
month + history,0.7876,+0.0020,0.0054,+0.4,4/8
month (Ludovico),0.7856,+0.0000,-,-,ref
month + next_peak,0.7842,-0.0015,0.0038,-0.4,5/8
next_peak instead of month,0.7839,-0.0017,0.0043,-0.4,5/8
month + weekday,0.7809,-0.0047,0.0025,-1.9,2/8


# <font color='#E8800A'>4. Effect on the official validation</font> <a class="anchor" id="official"></a>
[Back to TOC](#toc)

**The two submitted pipelines, with and without the new groups**, on the official scheme (3 yearly folds, 90-day gap) and with the Kaggle metric:
- V03: logistic regression + L1 selection (`C = 0.05`), Kaggle 0.804;
- V04: gradient boosting with default settings, Kaggle 0.796.

The honest F1 uses the threshold chosen on folds 1–2 and is scored on fold 3, as in every notebook so far.

In [12]:
l1_selector = SelectFromModel(
    make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=0.05, max_iter=5000)),
    importance_getter="named_steps.logisticregression.coef_",
)
SUBMITTED = {"V03 logistic + L1": "logistic regression", "V04 boosting": "gradient boosting"}


def submitted_pipeline(model_name, groups):
    """V03 (logistic + L1) or V04 (boosting) with the given feature groups."""
    steps = feature_pipeline({"groups": groups}, model_name).steps
    if model_name == "logistic regression":
        return Pipeline(steps[:-1] + [("select", clone(l1_selector)), steps[-1]])
    return Pipeline(steps)


def official_summary_full(pipeline):
    """official_summary of preprocessing_ludovico, on the frame with identifiers and date."""
    scores, oof = evaluate(pipeline, official_folds, X_full, y)
    early = np.concatenate([va for _, va in official_folds[:-1]])
    last = official_folds[-1][1]
    t_early, _ = best_threshold(y.iloc[early], oof.iloc[early])
    return {"ROC AUC mean": scores["ROC AUC"].mean(),
            "ROC AUC fold 1 / 2 / 3": " / ".join(f"{v:.3f}" for v in scores["ROC AUC"]),
            "threshold (folds 1-2)": t_early,
            "honest F1 (fold 3)": f1_score(y.iloc[last], oof.iloc[last] >= t_early)}


official = pd.DataFrame({
    (label, version): official_summary_full(submitted_pipeline(model_name, config["groups"]))
    for label, model_name in SUBMITTED.items()
    for version, config in configs.items()
}).T
official.style.format({"ROC AUC mean": "{:.4f}", "threshold (folds 1-2)": "{:.2f}", "honest F1 (fold 3)": "{:.3f}"})

# <font color='#E8800A'>5. From ROC AUC to F1: the threshold</font> <a class="anchor" id="threshold"></a>
[Back to TOC](#toc)

**Why this section:** ROC AUC measures how well the model **orders** the purchases (most likely to repeat first). F1 also depends on **where we cut** that order: every probability above the threshold becomes a 1. A better order is wasted if the cut lands in the wrong place.

**The honest F1 of section 4 is fragile:** one threshold, chosen on folds 1–2, scored on a single fold of ~830 rows. A more stable version uses the 8 half-year folds: for each fold from the 3rd on, the threshold is chosen **only on the folds before it** and the F1 is measured on it. That gives 6 honest scores instead of 1.

**Threshold rules compared** (all chosen on the earlier folds only):

| Rule | How the threshold of the next fold is set |
|---|---|
| `best F1` | the probability with the highest F1 on the earlier folds (the rule used so far) |
| `best F1, last 2 folds` | the same on the two most recent earlier folds: the positive rate drifts by year, and recent years are closer to the next one |
| `same share of 1s` | the **share of rows predicted 1** that gave the best F1 on the earlier folds; on the next fold the threshold is the probability that gives that share. It does not need the probabilities to keep the same scale from one fold to the next |

`best possible` (the best threshold of each fold, chosen on the fold itself) is the ceiling: no rule can beat it. `always 1` is the floor.

In [13]:
def rolling_thresholds(proba, folds, first=2):
    """F1 of each fold from `first` on, with the threshold set by each rule on the earlier folds only."""
    scores = []
    for k in range(first, len(folds)):
        earlier = np.concatenate([va for _, va in folds[:k]])
        recent = np.concatenate([va for _, va in folds[max(0, k - 2):k]])
        current = folds[k][1]
        y_now, p_now = y.iloc[current], proba.iloc[current]
        t_all, _ = best_threshold(y.iloc[earlier], proba.iloc[earlier])
        t_recent, _ = best_threshold(y.iloc[recent], proba.iloc[recent])
        share = (proba.iloc[earlier] >= t_all).mean()
        t_share = np.quantile(p_now, 1 - share)
        _, curve = best_threshold(y_now, p_now)
        scores.append({
            "fold": k + 1,
            "best F1": f1_score(y_now, p_now >= t_all),
            "best F1, last 2 folds": f1_score(y_now, p_now >= t_recent),
            "same share of 1s": f1_score(y_now, p_now >= t_share),
            "best possible": curve.max(),
            "always 1": f1_score(y_now, np.ones(len(y_now))),
        })
    return pd.DataFrame(scores).set_index("fold")


CANDIDATES = {(label, version): submitted_pipeline(model_name, configs[version]["groups"])
              for label, model_name in SUBMITTED.items()
              for version in [REFERENCE, "month + seasonal", "month + seasonal + weekday"]}
rolling = {}
for key, pipeline in CANDIDATES.items():
    _, oof_8 = evaluate(pipeline, prep_folds, X_full, y)
    rolling[key] = rolling_thresholds(oof_8, prep_folds)

RULES = ["best F1", "best F1, last 2 folds", "same share of 1s"]
threshold_board = pd.DataFrame({key: table.mean() for key, table in rolling.items()}).T
threshold_board.style.format("{:.4f}").highlight_max(axis=1, subset=RULES, color="#FCE4C4")

In [14]:
# Each rule minus the current one ('best F1'), fold by fold: mean, paired SEM, folds won
pd.DataFrame({
    (*key, rule): {"mean diff": (diff := table[rule] - table["best F1"]).mean(), "paired SEM": diff.sem(),
                   "folds won": f"{(diff > 0).sum()}/{len(diff)}"}
    for key, table in rolling.items() for rule in RULES[1:]
}).T

mean diff  \
V03 logistic + L1 month (Ludovico)           best F1, last 2 folds -0.001461   
                                             same share of 1s       0.000044   
                  month + seasonal           best F1, last 2 folds  -0.00204   
                                             same share of 1s       0.001337   
                  month + seasonal + weekday best F1, last 2 folds  0.004634   
                                             same share of 1s      -0.006382   
V04 boosting      month (Ludovico)           best F1, last 2 folds -0.003335   
                                             same share of 1s       -0.01155   
                  month + seasonal           best F1, last 2 folds -0.004143   
                                             same share of 1s      -0.002457   
                  month + seasonal + weekday best F1, last 2 folds -0.013989   
                                             same share of 1s      -0.004412   

                                                                   paired SEM  \
V03 logistic + L1 month (Ludovico)           best F1, last 2 folds    0.00247   
                                             same share of 1s        0.008253   
                  month + seasonal           best F1, last 2 folds   0.004236   
                                             same share of 1s        0.004739   
                  month + seasonal + weekday best F1, last 2 folds    0.00296   
                                             same share of 1s        0.008648   
V04 boosting      month (Ludovico)           best F1, last 2 folds   0.002845   
                                             same share of 1s        0.006311   
                  month + seasonal           best F1, last 2 folds   0.005666   
                                             same share of 1s        0.010528   
                  month + seasonal + weekday best F1, last 2 folds   0.011682   
                                             same share of 1s         0.00461   

                                                                   folds won  
V03 logistic + L1 month (Ludovico)           best F1, last 2 folds       1/6  
                                             same share of 1s            3/6  
                  month + seasonal           best F1, last 2 folds       1/6  
                                             same share of 1s            3/6  
                  month + seasonal + weekday best F1, last 2 folds       3/6  
                                             same share of 1s            2/6  
V04 boosting      month (Ludovico)           best F1, last 2 folds       0/6  
                                             same share of 1s            1/6  
                  month + seasonal           best F1, last 2 folds       1/6  
                                             same share of 1s            2/6  
                  month + seasonal + weekday best F1, last 2 folds       1/6  
                                             same share of 1s            1/6

**Do the feature groups help the F1 itself?** With the current rule (`best F1` on the earlier folds), each feature set minus the reference, fold by fold.

In [15]:
pd.DataFrame({
    (label, version): {"rolling honest F1": rolling[label, version]["best F1"].mean(),
                       "diff vs ref": (diff := rolling[label, version]["best F1"] - rolling[label, REFERENCE]["best F1"]).mean(),
                       "paired SEM": diff.sem(), "folds won": f"{(diff > 0).sum()}/{len(diff)}"}
    for label, version in CANDIDATES if version != REFERENCE
}).T

rolling honest F1 diff vs ref  \
V03 logistic + L1 month + seasonal                    0.759924    0.007476   
                  month + seasonal + weekday          0.761322    0.008875   
V04 boosting      month + seasonal                    0.741754   -0.008668   
                  month + seasonal + weekday          0.747246   -0.003176   

                                             paired SEM folds won  
V03 logistic + L1 month + seasonal             0.005959       4/6  
                  month + seasonal + weekday   0.007806       4/6  
V04 boosting      month + seasonal             0.007981       2/6  
                  month + seasonal + weekday   0.006632       1/6

# <font color='#E8800A'>6. Summary</font> <a class="anchor" id="summary"></a>
[Back to TOC](#toc)

### What each new group adds (reference: every column + the month, `features_ludovico`)

| Group | Logistic regression, 8 folds (ROC AUC) | Boosting, 8 folds (ROC AUC) | Official folds, V03 pipeline (ROC AUC) | Decision |
|---|---|---|---|---|
| `history` (share of past pauses ≤ 90 days, …) | +0.0011, 5/8 | +0.0020, 4/8 | +0.0007 | not kept: inside the noise |
| `next_peak` (days to the next peak month) | −0.0006 with the month, −0.0040 instead of it | −0.0015 / −0.0017 | lower | not kept |
| **`seasonal`** (the customer's own window one year earlier) | **+0.0037 ± 0.0021, 7/8** | +0.0026, 6/8 | **+0.0056, 3/3 folds up** | **kept** for linear models |
| `weekday` | +0.0016, 5/8 | −0.0047, 2/8 | +0.0027 | only next to `seasonal` (below) |
| `seasonal` + `weekday` | **+0.0052 ± 0.0026 (2.0 SEM), 6/8** | +0.0074 ± 0.0059, 6/8 | **+0.0073, 3/3 folds up** | candidate for V05 |

- **The customer's own seasonality is the first new information since the month.** The repeat rate goes from 0.32 for customers who never bought in that window in the last years to 0.76 for those who always did (§2). Unlike the pause history, the models had no column that carried it: `purchase_count_*` look at the *last* 30–365 days, not at *the same weeks one year earlier*.
- **History and `next_peak` do not help**, for the reasons in §2–3: the share of short pauses is already summarised by the recent purchase counts, and one number cannot replace the 12 month levels.
- **Gradient boosting with default settings gains in ROC AUC, not in F1** (§5). It stays below logistic regression in every table; it needs tuning (`06_models`) before its feature set can be judged.

### From ROC AUC to F1 (§5)

- **The honest F1 on fold 3 alone was misleading.** It dropped with `seasonal` (0.802 → 0.789) because the threshold chosen on folds 1–2 moved. Measured on **6 folds**, each with its threshold chosen on the folds before it, `seasonal` **raises the F1 of logistic regression by +0.0075 (4/6 folds)**, and `seasonal` + `weekday` by **+0.0089 (4/6)**. Both gains are positive but within ~1 SEM: the ROC AUC gain is the firmer evidence.
- **No threshold rule beats the current one.** Choosing it on the last 2 folds only, or by the share of 1s, changes the F1 by −0.014 to +0.005, never consistently. The current rule (best F1 on all earlier out-of-fold predictions) stays.
- **The threshold costs about 0.015 F1** against the best threshold of each fold chosen afterwards (`best possible`). That is the ceiling of any threshold work; the rest must come from better probabilities.

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Add the customer's own seasonality: purchases in the 90 days after the same date 1, 2, 3 years earlier (`seasonal`, 3 columns) | no seasonal feature; history of pauses; days to the next peak month | `features_nico` §3–4: logistic +0.0037 ± 0.0021 ROC AUC (7/8 folds), official 0.796 → 0.801 (3/3 up); rolling honest F1 +0.0075 (4/6) |
| Do not add purchase-history features read from the earlier rows (share of pauses ≤ 90 days, median, recent mean) | `history` group | §3–4: +0.001 / +0.002 ROC AUC, under 1 SEM |
| Do not replace or complement the month with the days to the next peak month | `days_to_next_peak` alone or with the month | §3–4: −0.001 to −0.006 ROC AUC |
| A pause or window read from the rows counts only if the data are complete there (pause = `days_since_previous_purchase`; no overlap with 2025-02-15 → 2025-05-24) | trust every row-based value | §1: the measured pause is right for only 47% of the customers' first test purchase |
| Keep the threshold rule: best F1 on all earlier out-of-fold predictions | best F1 on the last 2 folds; same share of predicted 1s | §5: neither alternative better on 6 rolling folds (−0.014 to +0.005) |